In [ ]:
import pandas as pd
import time
import zipfile
from tqdm.auto import tqdm
import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted
from collections import Counter

# ==========================================
# 1. CẤU HÌNH API GEMINI FLASH 3.6
# ==========================================
GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)

# Quay lại vũ khí thực dụng nhất, đủ quota, tốc độ bàn thờ
model = genai.GenerativeModel('gemini-3.6-flash')

# ==========================================
# 2. CONTRASTIVE FEW-SHOT PROMPT (ĐÃ CÂN BẰNG)
# ==========================================
def build_flash_few_shot_prompt(tweet_text, target):
    prompt = f"""You are an elite expert in Saudi sociolinguistics, Khaleeji dialects, and Twitter sentiment analysis.
Classify the EXACT stance of the following tweet regarding the topic: '{target}'.

RULES:
1. Output ONLY ONE WORD: 'Favor', 'Against', or 'None'.
2. SARCASM ALERT: Saudi Twitter frequently uses sarcasm. If a tweet seemingly praises women driving but actually mocks their driving skills, predicts accidents, or uses laughing emojis ironically, it is strictly 'Against'.
3. RELIGIOUS/SOCIAL OBJECTION: Claims that women driving causes corruption or mixes genders is strictly 'Against'.
4. GENUINE SUPPORT: Joy, relief, personal milestones, or defending the royal decree is 'Favor'.

CONTRASTIVE EXAMPLES:

[Example Pair 1: Sarcasm vs. Genuine Joy]
Tweet A: "وأخيراً بنسوق سياراتنا ونفتك من ذل السواقين شكرا لملكنا"
Stance A: Favor (Genuine relief and gratitude)
Tweet B: "إي خلهم يسوقون عشان نشوف حوادث تضحك في الشوارع ونوسع صدورنا 😂"
Stance B: Against (Sarcastic, mocking their skills, anticipating accidents)

[Example Pair 2: Religious/Social Fear vs. Neutral News]
Tweet A: "المرأة مكانها بيتها القيادة بتفتح أبواب الفساد والاختلاط"
Stance A: Against (Ideological opposition)
Tweet B: "أعلن المرور السعودي عن بدء استبدال الرخص الدولية برخص سعودية للنساء"
Stance B: None (Objective news reporting)

[Example Pair 3: Implicit Mocking]
Tweet A: "جهزوا التأمين الشامل ياشباب، الشوارع بتصير ملاهي سيارات تصادم"
Stance A: Against (Implicitly mocking women as bad drivers using humor)

[Example Pair 4: Personal Story — Genuine Favor]
Tweet A: "اليوم أول مرة أوصل أولادي المدرسة بسيارتي، شعور ما يوصف، أخيراً حرة"
Stance A: Favor (Personal milestone, genuine emotion of freedom and pride)

Now, deeply analyze and classify this new tweet:
Tweet: "{tweet_text}"
Stance:"""
    return prompt

def get_stance_flash(tweet_text, target, max_retries=5):
    prompt = build_flash_few_shot_prompt(tweet_text, target)
    
    for attempt in range(max_retries):
        try:
            # Set temperature = 0.0 để loại bỏ tính random, ép Flash trả lời logic nhất
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.0, 
                )
            )
            output = response.text.strip().capitalize()
            
            if "Favor" in output: return "Favor"
            elif "Against" in output: return "Against"
            elif "None" in output: return "None"
            else: return "None"
                
        except ResourceExhausted:
            wait_time = 4
            print(f"⚠️ Rate limit hit. Waiting {wait_time}s...")
            time.sleep(wait_time)
        except Exception as e:
            print(f"⚠️ API Error: {e}. Retrying...")
            time.sleep(2)
            
    return "None"

# ==========================================
# 3. THỰC THI TRÊN TẬP TEST
# ==========================================
print("Đang nạp tập Test...")
test_df = pd.read_csv("../data/test.csv")

predicted_labels = []

print(f"🚀 Bắt đầu Flash 3.6 Few-shot Inference trên {len(test_df)} mẫu...")
for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    tweet = row['tweet_text']
    target = row['target']
    
    pred_label = get_stance_flash(tweet, target)
    predicted_labels.append(pred_label)
    
    time.sleep(1) # Flash xử lý nhanh và quota lớn nên chỉ cần sleep 1s là đủ

# ==========================================
# 4. HẬU XỬ LÝ (DUMP NONE) VÀ ĐÓNG GÓI
# ==========================================
print("\n=== KẾT QUẢ VÀ ĐÓNG GÓI ===")
print(f"Raw distribution: {dict(Counter(predicted_labels))}")

# Lưu cả 2 phiên bản (Raw) ra TXT và ZIP
for name, preds in [("flash_raw_withnone", predicted_labels)]:
    txt = f"submission_{name}.txt"
    zip_name = f"submission_{name}.zip"
    
    with open(txt, "w", encoding="utf-8") as f:
        for l in preds:
            f.write(l + "\n")
            
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(txt)
        
    print(f"✅ Saved: {zip_name}")

In [2]:
import pandas as pd

train = pd.read_parquet("../data/StanceX_train.parquet")
test = pd.read_parquet("../data/StanceX_test.parquet")

train_wd = train[train["topic"].str.contains("women driving", case=False, na=False)]
test_wd = test[test["topic"].str.contains("women driving", case=False, na=False)]

train_wd.to_csv("../data/women_driving_train.csv", index=False, encoding="utf-8-sig")
test_wd.to_csv("../data/women_driving_test.csv", index=False, encoding="utf-8-sig")

print(len(train_wd), len(test_wd))

756 191


In [2]:
import pandas as pd

print("🚀 BẮT ĐẦU QUÁ TRÌNH TỔNG HỢP VÀ LỌC DỮ LIỆU (BAO GỒM DEV)...")

CONFIDENCE_THRESHOLD = 0.8
TARGET_TOPIC = 'empowerment'

# ==========================================
# 1. EXTERNAL DATA (WOMEN DRIVING)
# ==========================================
print("\n1. Đang load External Data...")
ext_train = pd.read_csv("../data/women_driving_train.csv")
ext_test  = pd.read_csv("../data/women_driving_test.csv")

ext_train.rename(columns={'topic': 'target'}, inplace=True)
ext_test.rename(columns={'topic': 'target'}, inplace=True)

gold_ext_df = pd.concat(
    [ext_train[['text', 'target', 'stance']],
     ext_test[['text', 'target', 'stance']]],
    ignore_index=True
)

# Giữ NaN rows: gán thành "None" thay vì drop
n_nan = gold_ext_df['stance'].isna().sum()
gold_ext_df['stance'] = gold_ext_df['stance'].fillna('None')
print(f"-> Tổng External: {len(gold_ext_df)} ({n_nan} NaN → gán 'None')")

# ==========================================
# 2. WOMEN EMPOWERMENT (TRAIN + DEV GỐC)
# ==========================================
print(f"\n2. Đang lọc Women Empowerment từ Train & Dev...")

def process_original_data(filepath):
    try:
        df = pd.read_csv(filepath, keep_default_na=False)
        df['stance:confidence'] = pd.to_numeric(df['stance:confidence'], errors='coerce')
        return df[
            (df['stance:confidence'] >= CONFIDENCE_THRESHOLD) &
            (df['target'].str.contains(TARGET_TOPIC, case=False, na=False))
        ][['text', 'target', 'stance']]
    except FileNotFoundError:
        print(f"⚠️ Không tìm thấy {filepath}")
        return pd.DataFrame(columns=['text', 'target', 'stance'])

empowerment_train_df = process_original_data("../data/train.csv")
empowerment_dev_df   = process_original_data("../data/dev.csv")
empowerment_df = pd.concat([empowerment_train_df, empowerment_dev_df], ignore_index=True)

print(f"-> Train: {len(empowerment_train_df)} | Dev: {len(empowerment_dev_df)}")
print(f"-> Tổng Empowerment: {len(empowerment_df)}")

# ==========================================
# 3. PSEUDO-LABELS TỪ TEST SET (GIỮ NGUYÊN None)
# ==========================================
print("\n3. Đang ghép Pseudo-labels...")
test_df = pd.read_csv("../data/test.csv")

with open("../data/submission_flash_raw_withnone.txt", "r", encoding="utf-8") as f:
    pseudo_labels = [line.strip() for line in f.readlines()]

test_df['stance'] = pseudo_labels
pseudo_df = test_df.rename(columns={'tweet_text': 'text'})[['text', 'target', 'stance']]
print(f"-> Pseudo-label: {len(pseudo_df)} mẫu (giữ nguyên None)")

# ==========================================
# 4. GỘP & KIỂM TRA
# ==========================================
print("\n4. Gộp Data và Shuffle...")
final_train_df = pd.concat([gold_ext_df, empowerment_df, pseudo_df], ignore_index=True)

# Capitalize để đồng nhất format
final_train_df['stance'] = final_train_df['stance'].str.strip().str.capitalize()

# Chỉ drop nếu text bị null, KHÔNG drop vì stance
final_train_df.dropna(subset=['text'], inplace=True)

# Validate: đảm bảo không còn label ngoài schema
valid_labels = {'Against', 'Favor', 'None'}
invalid_mask = ~final_train_df['stance'].isin(valid_labels)
if invalid_mask.sum() > 0:
    print(f"⚠️ {invalid_mask.sum()} rows có nhãn không hợp lệ → drop")
    final_train_df = final_train_df[~invalid_mask]

final_train_df = final_train_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n-> TỔNG SỐ MẪU: {len(final_train_df)}")
print("\nPhân phối nhãn:")
print(final_train_df['stance'].value_counts())
print("\nPhân phối theo source (target):")
print(final_train_df['target'].value_counts())

# ==========================================
# 5. LƯU
# ==========================================
output_path = "../data/final_augmented_train.csv"
final_train_df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"\n✅ Đã lưu: {output_path}")

🚀 BẮT ĐẦU QUÁ TRÌNH TỔNG HỢP VÀ LỌC DỮ LIỆU (BAO GỒM DEV)...

1. Đang load External Data...
-> Tổng External: 947 (146 NaN → gán 'None')

2. Đang lọc Women Empowerment từ Train & Dev...
-> Train: 782 | Dev: 139
-> Tổng Empowerment: 921

3. Đang ghép Pseudo-labels...
-> Pseudo-label: 352 mẫu (giữ nguyên None)

4. Gộp Data và Shuffle...

-> TỔNG SỐ MẪU: 2220

Phân phối nhãn:
stance
Favor      1293
Against     760
None        167
Name: count, dtype: int64

Phân phối theo source (target):
target
women driving        947
Women empowerment    921
Women Driving        352
Name: count, dtype: int64

✅ Đã lưu: ../data/final_augmented_train.csv


In [4]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import re, os, shutil, zipfile
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.model_selection import StratifiedKFold
from datasets import Dataset
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
from collections import Counter

# ==========================================
# 1. CẤU HÌNH CƠ BẢN & MARBERTv2
# ==========================================
# Nâng cấp lên MARBERTv2 (hiệu năng vượt trội MARBERTv1 trên các task phân loại)
MODEL_NAME    = "../model/marbert_base" 
tokenizer     = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

STANCE2ID    = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL     = {0: "Against", 1: "Favor", 2: "None"}
SENTIMENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARCASM2ID   = {"No": 0, "Yes": 1}

SEEDS = [42] 
N_SPLITS = 5 # Giảm xuống 5 để tiết kiệm thời gian compute với tập data lớn hơn, 10 folds đôi khi gây overfitting vào validation set.
SWA_K = 3

def clean_arabic_tweet(text):
    if not isinstance(text, str): return ""
    text = re.sub(r"http\S+|www\.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\u0640", "", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"(.)\1+", r"\1\1", text)
    return re.sub(r"\s+", " ", text.replace("#", " ")).strip()

# ==========================================
# 2. NẠP DỮ LIỆU ĐÃ ĐƯỢC AUGMENTED & MERGE AUXILIARY LABELS
# ==========================================
print("Đang nạp tập dữ liệu Augmented và Test...")

# Tập huấn luyện chính từ Pipeline trước đó
train_augmented = pd.read_csv("../data/final_augmented_train.csv")

# Tải lại train.csv và dev.csv gốc để map lại nhãn sentiment/sarcasm (nếu có)
orig_train = pd.read_csv("../data/train.csv", keep_default_na=False)
orig_dev   = pd.read_csv("../data/dev.csv", keep_default_na=False)
orig_full  = pd.concat([orig_train, orig_dev], ignore_index=True)[['text', 'sentiment', 'sarcasm']].drop_duplicates(subset=['text'])

# Merge để lấy auxiliary labels. Những câu từ External/Pseudo sẽ có giá trị NaN ở cột sentiment/sarcasm
full_train_df = pd.merge(train_augmented, orig_full, on='text', how='left')

# Đọc tập test để đánh giá cuối cùng
test_df = pd.read_csv("../data/test.csv", keep_default_na=False)
test_df.rename(columns={"tweet_text": "text"}, inplace=True) 

# Tiền xử lý Text
full_train_df["clean_text"] = full_train_df["text"].apply(clean_arabic_tweet)
test_df["clean_text"]       = test_df["text"].apply(clean_arabic_tweet)

# Encode Labels. ĐIỂM TỐI ƯU: Dùng -100 cho các nhãn bị khuyết để CrossEntropyLoss bỏ qua
full_train_df["label_stance"]    = full_train_df["stance"].map(STANCE2ID).fillna(2).astype(int)
full_train_df["label_sentiment"] = full_train_df["sentiment"].map(SENTIMENT2ID).fillna(-100).astype(int)
full_train_df["label_sarcasm"]   = full_train_df["sarcasm"].map(SARCASM2ID).fillna(-100).astype(int)

def tokenize_func(examples):
    # Target-aware attention: Nối target và text bằng [SEP]
    tokenized = tokenizer(examples["target"], examples["clean_text"], 
                          padding="max_length", truncation=True, max_length=128)
    if "label_stance" in examples:
        tokenized["labels_stance"]    = examples["label_stance"]
        tokenized["labels_sentiment"] = examples["label_sentiment"]
        tokenized["labels_sarcasm"]   = examples["label_sarcasm"]
    return tokenized

# ==========================================
# 3. MÔ HÌNH VÀ R-DROP TRAINER
# ==========================================
class MultiTaskMARBERTv2(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.bert           = AutoModel.from_pretrained(model_name)
        h                   = self.bert.config.hidden_size
        self.dropout        = nn.Dropout(0.2)
        # Khởi tạo trọng số đặc thù cho từng head
        self.stance_head    = nn.Linear(h, 3)
        self.sentiment_head = nn.Linear(h, 3)
        self.sarcasm_head   = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled = self.dropout(out.pooler_output)
        return self.stance_head(pooled), self.sentiment_head(pooled), self.sarcasm_head(pooled)

class RDropMultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_stance    = inputs.pop("labels_stance")
        labels_sentiment = inputs.pop("labels_sentiment")
        labels_sarcasm   = inputs.pop("labels_sarcasm")

        logits_s1, logits_se1, logits_sa1 = model(**inputs)
        logits_s2, logits_se2, logits_sa2 = model(**inputs)

        # ==========================================
        # TRỌNG SỐ CHO WEIGHTED CROSS-ENTROPY
        # Index 0: Against (2.0) | Index 1: Favor (2.0) | Index 2: None (0.5)
        # ==========================================
        weights = torch.tensor([2.0, 2.0, 0.5], dtype=torch.float).to(labels_stance.device)
        
        # Áp dụng weights vào hàm loss của nhánh Stance chính
        loss_fct_stance = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)
        
        # Nhánh phụ vẫn dùng ignore_index=-100
        loss_fct_aux = nn.CrossEntropyLoss(ignore_index=-100) 

        ce_loss = (
            loss_fct_stance(logits_s1, labels_stance)
            + 0.1  * loss_fct_aux(logits_se1, labels_sentiment)
            + 0.05 * loss_fct_aux(logits_sa1, labels_sarcasm)
        )

        def sym_kl(l1, l2):
            p1, p2 = F.log_softmax(l1, dim=-1), F.log_softmax(l2, dim=-1)
            q1, q2 = F.softmax(l1, dim=-1), F.softmax(l2, dim=-1)
            return (F.kl_div(p1, q2, reduction="batchmean") + F.kl_div(p2, q1, reduction="batchmean")) / 2.0

        kl_loss = sym_kl(logits_s1, logits_s2)
        total_loss = ce_loss + 0.5 * kl_loss

        return (total_loss, {"logits_stance": logits_s1}) if return_outputs else total_loss

def swa_checkpoint_averaging(model, checkpoint_dir, k=3):
    all_ckpts = sorted([d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")], key=lambda x: int(x.split("-")[-1]))
    selected = all_ckpts[-k:]
    if not selected: return model

    avg_state = None
    count = 0
    for ckpt_name in selected:
        st_path = os.path.join(checkpoint_dir, ckpt_name, "model.safetensors")
        if not os.path.exists(st_path): continue
        from safetensors.torch import load_file
        raw = load_file(st_path, device="cpu")
        state = {k: v.float() for k, v in raw.items()}
        if avg_state is None: avg_state = state
        else:
            for key in avg_state: avg_state[key] += state[key]
        count += 1

    for key in avg_state: avg_state[key] /= count
    model.load_state_dict(avg_state, strict=True)
    return model

# ==========================================
# 4. TRAINING & MEGA-ENSEMBLE
# ==========================================
cols_data_train = ["target", "clean_text", "label_stance", "label_sentiment", "label_sarcasm"]
cols_data_test  = ["target", "clean_text"]

test_ds = Dataset.from_pandas(test_df[cols_data_test]).map(tokenize_func, batched=True).remove_columns(cols_data_test)
test_loader = DataLoader(test_ds.with_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids"]), batch_size=32, shuffle=False)

mega_test_probs = np.zeros((len(test_df), 3))
total_models = len(SEEDS) * N_SPLITS

# Cân bằng fold dựa trên cả target và stance để phân phối đều dữ liệu "Women driving"
strat_key = full_train_df["target"] + "_" + full_train_df["label_stance"].astype(str)

for seed in SEEDS:
    print(f"\n{'='*50}\n🚀 BẮT ĐẦU HUẤN LUYỆN VỚI SEED {seed}\n{'='*50}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for fold, (train_idx, val_idx) in enumerate(skf.split(full_train_df, strat_key)):
        print(f"\n--- SEED {seed} | FOLD {fold+1}/{N_SPLITS} ---")
        
        fold_train_df = full_train_df.iloc[train_idx]
        fold_train_ds = Dataset.from_pandas(fold_train_df[cols_data_train]).map(tokenize_func, batched=True).remove_columns(cols_data_train)

        fold_dir = f"../model/marbertv2_rdrop_swa_s{seed}_f{fold}"
        model = MultiTaskMARBERTv2(MODEL_NAME)

        args = TrainingArguments(
            output_dir                  = fold_dir,
            learning_rate               = 2e-5,
            per_device_train_batch_size = 16,
            num_train_epochs            = 5, # Có thể giảm xuống 4 vì MARBERTv2 hội tụ rất nhanh
            lr_scheduler_type           = "cosine",
            warmup_ratio                = 0.1,
            weight_decay                = 0.01,
            bf16                        = True, # Chạy bf16 nếu dùng GPU Ampere trở lên
            seed                        = seed,
            eval_strategy               = "no",
            save_strategy               = "epoch",
            save_total_limit            = SWA_K,
            label_names                 = ["labels_stance", "labels_sentiment", "labels_sarcasm"],
            logging_steps               = 100,
            report_to                   = "none",
        )

        trainer = RDropMultiTaskTrainer(model=model, args=args, train_dataset=fold_train_ds, data_collator=data_collator)
        trainer.train()

        # Áp dụng SWA ngay khi kết thúc fold
        model = swa_checkpoint_averaging(model, fold_dir, k=SWA_K)
        shutil.rmtree(fold_dir, ignore_errors=True)

        model.eval().to(device)
        fold_test_probs = []
        with torch.no_grad():
            for batch in test_loader:
                out = model(input_ids=batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device), token_type_ids=batch.get("token_type_ids", torch.zeros_like(batch["input_ids"])).to(device))
                fold_test_probs.append(F.softmax(out[0], dim=-1).cpu().numpy())
        
        mega_test_probs += np.vstack(fold_test_probs)
        del model, trainer
        torch.cuda.empty_cache()

mega_test_probs /= total_models
np.save("marbertv2_augmented_probs.npy", mega_test_probs)
print("✅ Đã lưu Softmax Probabilities: marbertv2_augmented_probs.npy")

# ==========================================
# 5. LẤY KẾT QUẢ DỰ ĐOÁN CHUẨN (KHÔNG DÙNG DUMP NONE)
# ==========================================
print("\n=== LẤY KẾT QUẢ DỰ ĐOÁN TỪ ENSEMBLE ===")

standard_preds = []

for p in mega_test_probs:
    pred = np.argmax(p)  # Lấy trực tiếp class có xác suất cao nhất
    standard_preds.append(pred)

labels = [ID2LABEL[p] for p in standard_preds]
print(f"📊 Phân phối cuối cùng: {dict(Counter(labels))}")

out_name = "submission_marbertv2_augmented_standard"
with open(f"{out_name}.txt", "w", encoding="utf-8") as f:
    for l in labels: f.write(l + "\n")
        
with zipfile.ZipFile(f"{out_name}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(f"{out_name}.txt")

print(f"\n🎉 ĐÃ HOÀN TẤT! Hãy nộp ngay file '{out_name}.zip' lên Leaderboard!")

Đang nạp tập dữ liệu Augmented và Test...


Map: 100%|██████████| 352/352 [00:00<00:00, 9235.38 examples/s]



🚀 BẮT ĐẦU HUẤN LUYỆN VỚI SEED 42

--- SEED 42 | FOLD 1/5 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1863.49it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,0.854801
200,0.604224
300,0.473946
400,0.375715
500,0.346317


...

Step,Training Loss
100,0.853023
200,0.609593
300,0.485228
400,0.380903
500,0.340753



--- SEED 42 | FOLD 5/5 ---


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4144.40it/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
100,0.858976
200,0.565485
300,0.472179
400,0.378902
500,0.338673


✅ Đã lưu Softmax Probabilities: marbertv2_augmented_probs.npy

=== LẤY KẾT QUẢ DỰ ĐOÁN TỪ ENSEMBLE ===
📊 Phân phối cuối cùng: {'Against': 197, 'Favor': 155}

🎉 ĐÃ HOÀN TẤT! Hãy nộp ngay file 'submission_marbertv2_augmented_standard.zip' lên Leaderboard!
